# SWOT 분석 (데이터 기반)

## 목적
이 노트북은 내부 데이터(성과·운영·고객)와 외부 데이터(시장·경쟁·트렌드)를 결합해 **강점(S)·약점(W)·기회(O)·위협(T)** 요소를 정량/정성으로 도출하고, 최종적으로 **TOWS 전략( SO/ST/WO/WT )**까지 제안하는 것을 목표로 합니다.

## 사용 데이터 (CSV)
- 내부(Strength/Weakness)
  - `sales_data.csv`: 매출/수익 지표
  - `operational_metrics.csv`: 운영 효율/품질 지표
  - `customer_satisfaction.csv`: 고객만족/불만 지표
  - `customer_reviews.csv`: 고객 리뷰 텍스트(+ rating)
- 외부(Opportunity/Threat)
  - `market_data.csv`: 시장 규모/점유 등
  - `industry_growth.csv`: 산업 성장률
  - `competitor_analysis.csv`: 경쟁사 비교
  - `industry_satisfaction.csv`: 산업/시장 만족도
  - `consumer_trends.csv`: 트렌드 영향(impact), 확산(adoption) 등
  - `website_analytics.csv`: 유입/전환 등 디지털 지표

## 방법(요약)
- 정량 지표: 표준화/스케일링(Min-Max) 후 기준별 점수화 → S/W/O/T 후보 산출
- 리뷰 텍스트: rating 기반 긍·부정 분리 → `KoNLPy(Okt)`로 토큰화 → `TF-IDF` + `NMF`로 토픽 추출(강점/약점 키워드)
- 트렌드: 관련성×영향력 매트릭스(시각화)로 기회/위협 분류(레이블 겹침은 `adjustText`로 보정)
- 전략: 도출된 S/W/O/T를 조합해 TOWS 전략을 생성하며, 필요 시 **OpenAI GPT(선택)**로 전략 문장/요약을 보강합니다.

## 참고
- 로컬 실행 시 데이터는 기본적으로 저장소의 `swot/` 폴더에서 로드합니다.
- OpenAI를 쓰려면 환경변수 `OPENAI_API_KEY`가 필요합니다(없어도 비-LLM 부분은 실행 가능).

In [ ]:
# 환경 설정 (Colab/로컬 공통)
# - 로컬(macOS/Windows): 이 셀은 설치를 건너뜁니다. 터미널에서 아래를 실행하세요.
#   - Windows: .venv\Scripts\python -m pip install -r swot\requirements.txt
#   - macOS/Linux: .venv/bin/python -m pip install -r swot/requirements.txt
# - Colab: 필요한 경우에만 자동 설치/폰트 설치를 수행합니다.

import os
import sys
import shutil
import subprocess
from pathlib import Path

# Matplotlib 캐시 디렉터리를 작업폴더로 고정(권장: 권한 이슈/속도 개선)
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


IN_COLAB = _in_colab()

if IN_COLAB:
    print("Colab 환경: 라이브러리 설치를 진행합니다...")

    req = Path("requirements.txt")
    if not req.exists():
        alt = Path("swot") / "requirements.txt"
        req = alt if alt.exists() else req

    if req.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    else:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "konlpy",
            "adjustText",
            "pandas",
            "seaborn",
            "matplotlib",
            "scikit-learn",
            "openpyxl",
            "numpy",
            "requests",
            "openai",
            "python-dotenv",
        ])

    # Colab에서 Nanum 폰트 설치 (로컬에서는 불필요)
    try:
        subprocess.check_call(
            [
                "bash",
                "-lc",
                "apt-get update -qq && apt-get install -y -qq fonts-nanum && fc-cache -fv >/dev/null",
            ]
        )
    except Exception as e:
        print(f"폰트 설치를 건너뜁니다: {e}")

    # Matplotlib 폰트 캐시 초기화
    try:
        shutil.rmtree(Path.home() / ".cache" / "matplotlib", ignore_errors=True)
    except Exception:
        pass

    print("Colab: 설치/폰트 설정 완료")
else:
    print("로컬 환경: 설치 셀을 건너뜁니다. 필요하면 requirements.txt를 설치하세요.")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# .env 파일 로드 (swot/ 폴더 또는 현재 디렉터리에서 검색)
for env_path in [Path('swot/.env'), Path('.env')]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env 파일 로드 완료: {env_path}")
        break

# Matplotlib 캐시 디렉터리를 작업폴더로 고정(권한 이슈/속도 개선)
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from konlpy.tag import Okt
from adjustText import adjust_text
import re
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from itertools import product
import requests  # 사용 안 함
from openai import OpenAI  # OpenAI 라이브러리 import
import time  # API 호출 간격 제어용
import warnings

# 경고 메시지 무시 (선택 사항)
warnings.filterwarnings('ignore')

# Matplotlib 기본 설정 (크로스플랫폼 폰트 fallback)
plt.rcParams["font.family"] = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 Import 및 Matplotlib 설정 완료.")


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


# --- OpenAI API 키 설정 ---
# 1) 로컬/서버 공통: 환경변수 OPENAI_API_KEY (.env 또는 시스템 환경변수)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# 2) Colab일 때만: userdata(비밀번호)에서 가져오기 시도
if (OPENAI_API_KEY is None) and _in_colab():
    try:
        from google.colab import userdata  # type: ignore

        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
        print("OpenAI API 키 로드 완료 (Colab 비밀번호 사용).")
    except Exception as e:
        print(f"Colab userdata에서 API 키를 가져오지 못했습니다: {e}")

if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI 클라이언트 초기화 완료.")
else:
    openai_client = None
    print("경고: OPENAI_API_KEY가 설정되지 않았습니다. OpenAI 호출 셀은 실패할 수 있습니다.")

In [ ]:
# --- 데이터 파일 경로 설정 ---
# 로컬 실행(권장): 저장소의 swot/ 폴더를 기본 데이터 경로로 사용
# Colab/Drive 사용 시: '/content/drive/MyDrive/...' 형태로 data_path를 바꿔주세요.

from pathlib import Path

if Path('swot').exists():
    data_path = Path('swot')
else:
    data_path = Path('.')

print(f"데이터 경로: {data_path.resolve()}")


# --- 파일 존재 확인 함수 ---
def check_file(path):
    """파일 존재 여부를 확인하고 결과를 반환합니다."""
    path = Path(path)
    if not path.exists():
        print(f"경고: 파일이 존재하지 않습니다 - {path}")
        return False
    print(f"파일 확인: {path}")
    return True


# --- 요소 데이터 가져오기 헬퍼 함수 ---
def get_element_data(df, condition_col, condition_val, score_col, extra_cols=[], default_score=0.0):
    """주어진 조건에 맞는 행의 점수와 추가 컬럼 값들을 가져옵니다. (NaN 처리 개선)"""
    default_result = {'score': default_score}
    for col in extra_cols:
        default_result[col] = np.nan  # 기본값은 NaN으로 설정

    if df is None or df.empty:
        return default_result
    try:
        filtered_df = df[df[condition_col] == condition_val]
        if filtered_df.empty:
            return default_result

        score_val = filtered_df[score_col].values[0]
        result = {'score': float(score_val) if pd.notna(score_val) else default_score}

        for col in extra_cols:
            if col in filtered_df.columns:
                val = filtered_df[col].values[0]
                result[col] = val if pd.notna(val) else np.nan
            else:
                result[col] = np.nan
        return result
    except Exception as e:
        print(f"Warning: Error retrieving data for {condition_val}. Error: {e}. Using defaults.")
        error_result = {'score': default_score}
        for col in extra_cols:
            error_result[col] = np.nan
        return error_result


print("데이터 경로 설정 및 헬퍼 함수 정의 완료.")

In [ ]:
# --- 판매 실적 분석 (내부 - 강점/약점) ---
print("\n--- 1. 판매 실적 분석 시작 ---")
sales_data_path = os.path.join(data_path, 'sales_data.csv')
industry_growth_path = os.path.join(data_path, 'industry_growth.csv')
sales_data = None
industry_growth = None
category_comparison = pd.DataFrame() # 빈 DataFrame으로 초기화

if check_file(sales_data_path) and check_file(industry_growth_path):
    sales_data = pd.read_csv(sales_data_path)
    print("판매 실적 데이터 로드 완료.")
    # print(sales_data.head()) # 데이터 확인용

    # 제품 카테고리별 성장률 계산 (데이터가 최소 2개 이상일 때만 계산)
    sales_growth = sales_data.groupby('category').filter(lambda x: len(x) >= 2).groupby('category').apply(
        lambda x: (x['sales'].iloc[-1] - x['sales'].iloc[0]) / x['sales'].iloc[0] * 100 if x['sales'].iloc[0] != 0 else 0
    ).reset_index(name='growth_rate')

    industry_growth = pd.read_csv(industry_growth_path)
    print("산업 평균 성장률 데이터 로드 완료.")
    # print(industry_growth.head()) # 데이터 확인용

    # 카테고리별 성장률 비교 분석
    category_comparison = pd.merge(sales_growth, industry_growth, on='category', how='left')
    category_comparison['industry_growth'] = category_comparison['industry_growth'].fillna(0) # NaN은 0으로 처리
    category_comparison['relative_growth'] = category_comparison['growth_rate'] - category_comparison['industry_growth']
    print("카테고리별 상대 성장률 계산 완료.")
    # print(category_comparison) # 결과 확인용

    # 시각화 (선택적)
    if not category_comparison.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(x='category', y='relative_growth', data=category_comparison)
        plt.axhline(y=0, color='r', linestyle='-')
        plt.title('카테고리별 산업 평균 대비 성장률')
        plt.ylabel('상대적 성장률 (%)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

else:
    print("판매 실적 또는 산업 성장률 데이터 파일을 찾을 수 없어 분석을 건너<0xEB><0x9A><0x8D>니다.")
print("--- 1. 판매 실적 분석 완료 ---")

In [ ]:
# --- 고객 만족도 분석 (내부 - 강점/약점) ---
print("\n--- 2. 고객 만족도 분석 시작 ---")
satisfaction_data_path = os.path.join(data_path, 'customer_satisfaction.csv')
industry_satisfaction_path = os.path.join(data_path, 'industry_satisfaction.csv')
satisfaction_data = None
industry_satisfaction = None
satisfaction_comparison = pd.DataFrame() # 초기화

if check_file(satisfaction_data_path) and check_file(industry_satisfaction_path):
    satisfaction_data = pd.read_csv(satisfaction_data_path)
    print("고객 만족도 데이터 로드 완료.")
    # print(satisfaction_data.head()) # 데이터 확인용

    satisfaction_matrix = satisfaction_data[['attribute', 'satisfaction_score', 'importance_score']]

    industry_satisfaction = pd.read_csv(industry_satisfaction_path)
    print("산업 평균 만족도 데이터 로드 완료.")
    # print(industry_satisfaction.head()) # 데이터 확인용

    # 속성별 만족도 비교 분석
    satisfaction_comparison = pd.merge(satisfaction_matrix, industry_satisfaction, on='attribute', how='left')
    # 산업 평균 만족도 없는 경우 자사 점수와 동일하게 간주 (격차 0) -> 수정: NaN 유지 후 gap 계산 시 처리
    # satisfaction_comparison['industry_avg'] = satisfaction_comparison['industry_avg'].fillna(satisfaction_comparison['satisfaction_score'])
    satisfaction_comparison['satisfaction_gap'] = satisfaction_comparison['satisfaction_score'] - satisfaction_comparison['industry_avg'] # NaN이 있다면 결과도 NaN
    print("속성별 만족도 격차 계산 완료.")
    # print(satisfaction_comparison) # 결과 확인용

    # 중요도-만족도 매트릭스 (IPA) 시각화 (선택적)
    if not satisfaction_comparison.dropna(subset=['satisfaction_score', 'importance_score']).empty: # 유효 데이터 있을 때만
        plt.figure(figsize=(10, 8))
        temp_df = satisfaction_comparison.dropna(subset=['satisfaction_score', 'importance_score']).copy() # NaN 제거 후 시각화
        mean_importance = temp_df['importance_score'].mean()
        mean_satisfaction = temp_df['satisfaction_score'].mean()

        sns.scatterplot(
            x='importance_score',
            y='satisfaction_score',
            size='importance_score', # 크기는 중요도로
            hue='satisfaction_gap', # 색상은 만족도 격차로 (NaN 값은 표시 안될 수 있음)
            palette='RdYlGn',
            data=temp_df,
            legend='brief', # 범례 간소화
            sizes=(50, 500)
        )
        plt.axhline(y=mean_satisfaction, color='gray', linestyle='--', alpha=0.7)
        plt.axvline(x=mean_importance, color='gray', linestyle='--', alpha=0.7)

        # 점 레이블 추가 (adjustText 사용)
        texts = []
        for i, row in temp_df.iterrows():
             texts.append(plt.text(row['importance_score'], row['satisfaction_score'], row['attribute'], fontsize=9))
        # adjust_text(texts, arrowprops=dict(arrowstyle="-", color='k', lw=0.5)) # 필요시 주석 해제, 레이블 많으면 오래 걸림

        plt.title('중요도-만족도 분석 (IPA)')
        plt.xlabel('중요도')
        plt.ylabel('만족도')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='만족도 격차', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
        print("IPA 시각화를 위한 유효 데이터가 부족합니다.")

else:
    print("고객 만족도 또는 산업 평균 만족도 데이터 파일을 찾을 수 없어 분석을 건너<0xEB><0x9A><0x8D>니다.")
print("--- 2. 고객 만족도 분석 완료 ---")

In [ ]:
# --- 시장 데이터 분석 (외부 - 기회/위협) ---
print("\n--- 3. 시장 데이터 분석 시작 ---")
market_data_path = os.path.join(data_path, 'market_data.csv')
market_data = None

if check_file(market_data_path):
    market_data = pd.read_csv(market_data_path)
    print("시장 데이터 로드 완료.")
    # print(market_data.head()) # 데이터 확인용

    # 시장 매력도-경쟁 강도 매트릭스 시각화 (선택적)
    if not market_data.dropna(subset=['market_growth', 'competition_intensity']).empty:
        plt.figure(figsize=(10, 8))
        temp_df = market_data.dropna(subset=['market_growth', 'competition_intensity', 'market_size']).copy()
        mean_competition = temp_df['competition_intensity'].mean()
        mean_growth = temp_df['market_growth'].mean()

        sns.scatterplot(
            x='competition_intensity',
            y='market_growth',
            size='market_size', # 크기는 시장 크기로
            hue='market_growth', # 색상은 시장 성장률로
            palette='YlGnBu',
            data=temp_df,
            legend='brief',
            sizes=(50, 500)
        )
        plt.axhline(y=mean_growth, color='gray', linestyle='--')
        plt.axvline(x=mean_competition, color='gray', linestyle='--')

        # 점 레이블 추가
        texts = []
        for i, row in temp_df.iterrows():
             texts.append(plt.text(row['competition_intensity'], row['market_growth'], row['category'], fontsize=9))
        # adjust_text(texts, arrowprops=dict(arrowstyle="-", color='k', lw=0.5)) # 필요시 주석 해제

        plt.title('시장 매력도-경쟁 강도 분석')
        plt.xlabel('경쟁 강도')
        plt.ylabel('시장 성장률(%)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='시장 성장률(%)', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
         print("시장 매력도 시각화를 위한 유효 데이터가 부족합니다.")

else:
    print("시장 데이터 파일을 찾을 수 없어 분석을 건너<0xEB><0x9A><0x8D>니다.")
print("--- 3. 시장 데이터 분석 완료 ---")

In [ ]:
# --- 소비자 트렌드 분석 (외부 - 기회/위협) ---
print("\n--- 4. 소비자 트렌드 분석 시작 ---")
trend_data_path = os.path.join(data_path, 'consumer_trends.csv')
trend_data = None
trend_analysis = pd.DataFrame() # 초기화

if check_file(trend_data_path):
    trend_data = pd.read_csv(trend_data_path)
    print("소비자 트렌드 데이터 로드 완료.")
    # print(trend_data.head()) # 데이터 확인용

    # 트렌드 영향력 및 확산 속도 분석
    trend_analysis = trend_data[['trend_name', 'impact_score', 'adoption_rate', 'relevance_to_business']].copy()

    # 긍정적 영향과 부정적 영향 분리
    trend_analysis['impact_type'] = trend_analysis['impact_score'].apply(
        lambda x: 'Positive' if x > 0 else ('Negative' if x < 0 else 'Neutral')
    )
    print("트렌드 영향 유형 분류 완료.")
    # print(trend_analysis) # 결과 확인용

    # 트렌드 영향력-관련성 매트릭스 시각화 (선택적)
    if not trend_analysis.dropna(subset=['impact_score', 'relevance_to_business']).empty:
        plt.figure(figsize=(12, 8))
        temp_df = trend_analysis.dropna(subset=['impact_score', 'relevance_to_business', 'adoption_rate']).copy()
        mean_relevance = temp_df['relevance_to_business'].mean()

        sns.scatterplot(
            x='relevance_to_business',
            y='impact_score',
            size='adoption_rate', # 크기는 확산 속도로
            hue='impact_type', # 색상은 영향 유형으로
            palette={'Positive': 'green', 'Negative': 'red', 'Neutral': 'grey'},
            sizes=(50, 400),
            data=temp_df,
            legend='brief'
        )
        plt.axhline(y=0, color='gray', linestyle='--')
        plt.axvline(x=mean_relevance, color='gray', linestyle='--')

        # 점 레이블 위치 조정 (adjustText 사용)
        texts_trend = []
        for i, row in temp_df.iterrows():
            texts_trend.append(
                plt.text(row['relevance_to_business'], row['impact_score'],
                         row['trend_name'], fontsize=9)
            )
        adjust_text(texts_trend, arrowprops=dict(arrowstyle='->', color='black', lw=0.5)) # 겹침 방지

        plt.title('소비자 트렌드 영향 분석')
        plt.xlabel('비즈니스 관련성')
        plt.ylabel('영향력 점수(긍정/부정)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='영향 유형', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
        print("트렌드 영향력 시각화를 위한 유효 데이터가 부족합니다.")

else:
    print("소비자 트렌드 데이터 파일을 찾을 수 없어 분석을 건너<0xEB><0x9A><0x8D>니다.")
print("--- 4. 소비자 트렌드 분석 완료 ---")

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from konlpy.tag import Okt
from adjustText import adjust_text
import re

# 한국어 형태소 분석기 초기화
okt = Okt()

# 한글 불용어 리스트 (예시)
korean_stopwords = [
    '정말', '너무', '같습니다', '있습니다', '합니다', '것이', '입니다', '에서', '으로', '에게',
    '제품', '점', '다음', '다른', '비교', '확실히', '같아요', '아니에요', '별로예요', '좀'
]

# 한글 텍스트 전처리 함수
def preprocess_text(text):
    # 1. 특수문자 및 숫자 제거, 한글과 공백만 유지
    text = re.sub(r'[^가-힣\s]', '', text)
    # 2. 공백 과다 제거 및 소문자 변환 불필요 (한글)
    text = ' '.join(text.split())
    return text

# 키워드 추출 함수 (2음절 이상, 명사/형용사/동사, 불용어 제거)
def extract_keywords(text):
    # 텍스트 전처리
    text = preprocess_text(text)
    # 형태소 분석 및 품사 태깅
    tokens = okt.pos(text, norm=True, stem=True)
    # 2음절 이상, 명사/형용사/동사, 불용어 제외
    keywords = [
        token for token, pos in tokens
        if pos in ['Noun'] and len(token) >= 2 and token not in korean_stopwords
    ]
    return keywords

# 고객 리뷰 데이터 로드
reviews_data = pd.read_csv(Path(data_path) / 'customer_reviews.csv')

# Rating 기반 감성 분석
reviews_data['sentiment'] = reviews_data['rating'].apply(
    lambda x: 'positive' if x >= 4 else 'negative'
)

print("감성 분석 결과 (Rating 기반):")
print(reviews_data['sentiment'].value_counts())

# 카테고리별 감성 분석
category_sentiment = reviews_data.groupby(['product_category', 'sentiment']).size().unstack()
category_sentiment = category_sentiment.fillna(0)
category_sentiment['positive_ratio'] = category_sentiment['positive'] / (
    category_sentiment['positive'] + category_sentiment['negative']
)

# 카테고리별 감성 비율 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=category_sentiment.index, y=category_sentiment['positive_ratio'])
plt.title('카테고리별 긍정 리뷰 비율')
plt.xlabel('제품 카테고리')
plt.ylabel('긍정 리뷰 비율')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 주요 토픽 추출을 위한 TF-IDF 분석 (전처리 및 조건 적용)
vectorizer = TfidfVectorizer(max_features=1000, tokenizer=extract_keywords)
tfidf = vectorizer.fit_transform(reviews_data[reviews_data['sentiment'] == 'positive']['review_text'])

# 주제 모델링(NMF) 적용
nmf_model = NMF(n_components=5, random_state=42)
nmf_features = nmf_model.fit_transform(tfidf)

# 각 토픽의 주요 키워드 추출
feature_names = vectorizer.get_feature_names_out()
strengths_topics = []

for topic_idx, topic in enumerate(nmf_model.components_):
    top_features_idx = topic.argsort()[:-11:-1]
    top_features = [feature_names[i] for i in top_features_idx]
    strengths_topics.append(f"강점 토픽 {topic_idx+1}: {', '.join(top_features)}")

print("\n강점 관련 주요 토픽:")
for topic in strengths_topics:
    print(topic)

# 부정 리뷰에서 토픽 추출 (약점)
tfidf = vectorizer.fit_transform(reviews_data[reviews_data['sentiment'] == 'negative']['review_text'])
nmf_model = NMF(n_components=5, random_state=42)
nmf_features = nmf_model.fit_transform(tfidf)

feature_names = vectorizer.get_feature_names_out()
weaknesses_topics = []

for topic_idx, topic in enumerate(nmf_model.components_):
    top_features_idx = topic.argsort()[:-11:-1]
    top_features = [feature_names[i] for i in top_features_idx]
    weaknesses_topics.append(f"약점 토픽 {topic_idx+1}: {', '.join(top_features)}")

print("\n약점 관련 주요 토픽:")
for topic in weaknesses_topics:
    print(topic)

# 소비자 트렌드 데이터 로드
trend_data = pd.read_csv(Path(data_path) / 'consumer_trends.csv')

# 트렌드 영향력 및 확산 속도 분석
trend_analysis = trend_data[['trend_name', 'impact_score', 'adoption_rate', 'relevance_to_business']]

# 긍정적 영향과 부정적 영향 분리
trend_analysis['impact_type'] = trend_analysis['impact_score'].apply(
    lambda x: 'Positive' if x > 0 else 'Negative'
)

# 트렌드 영향력-관련성 매트릭스 작성
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='relevance_to_business',
    y='impact_score',
    size='adoption_rate',
    hue='impact_type',
    palette={'Positive': 'green', 'Negative': 'red'},
    sizes=(50, 400),
    data=trend_analysis
)

plt.axhline(y=0, color='gray', linestyle='--')
plt.axvline(x=trend_analysis['relevance_to_business'].mean(), color='gray', linestyle='--')

# 점 레이블 위치 조정 (1사분면은 adjustText 사용)
mean_relevance = trend_analysis['relevance_to_business'].mean()
texts = []
for i, row in trend_analysis.iterrows():
    if (row['relevance_to_business'] > mean_relevance) and (row['impact_score'] > 0):  # 1사분면
        texts.append(
            plt.text(row['relevance_to_business'], row['impact_score'],
                     row['trend_name'], fontsize=9, ha='right', va='center')
        )
    else:  # 나머지 사분면
        plt.text(row['relevance_to_business'], row['impact_score'] - 0.05,
                 row['trend_name'], fontsize=9, ha='center', va='top')

# 1사분면 레이블 겹침 방지
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='black', lw=0.5))

# 범례 위치
plt.legend(title='영향 유형 및 확산 속도', loc='lower center', bbox_to_anchor=(0.8, -0.35),
           ncol=2, fontsize=10, title_fontsize=12)

plt.title('소비자 트렌드 영향 분석', fontsize=14, fontweight='bold')
plt.xlabel('비즈니스 관련성', fontsize=12)
plt.ylabel('영향력 점수(긍정/부정)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 기회와 위협 요소 식별
opportunities = trend_analysis[
    (trend_analysis['impact_score'] > 0) &
    (trend_analysis['relevance_to_business'] > trend_analysis['relevance_to_business'].mean())
]

threats = trend_analysis[
    (trend_analysis['impact_score'] < 0) &
    (trend_analysis['relevance_to_business'] > trend_analysis['relevance_to_business'].mean())
]

print("기회 요소(긍정적 영향, 높은 관련성):")
print(opportunities[['trend_name', 'impact_score', 'adoption_rate']])
print("\n위협 요소(부정적 영향, 높은 관련성):")
print(threats[['trend_name', 'impact_score', 'adoption_rate']])

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# 분석 결과를 통합하여 SWOT 요소 정리
swot_elements = {
    'strengths': [
        {'element': '모바일 전자기기 매출 성장률', 'score': float(category_comparison[category_comparison['category'] == '모바일 전자기기']['relative_growth'].values[0]), 'source': '판매 데이터'},
        {'element': '고객 서비스 만족도', 'score': float(satisfaction_comparison[satisfaction_comparison['attribute'] == '고객 서비스']['satisfaction_gap'].values[0]), 'source': '고객 만족도 조사'},
        {'element': '배송 속도 및 정확성', 'score': float(satisfaction_comparison[satisfaction_comparison['attribute'] == '배송']['satisfaction_gap'].values[0]), 'source': '고객 만족도 조사'},
        {'element': '사용자 친화적 웹사이트', 'score': 0.78, 'source': '고객 리뷰 감성 분석'}
    ],
    'weaknesses': [
        {'element': '가전제품 매출 성장률', 'score': float(category_comparison[category_comparison['category'] == '가전제품']['relative_growth'].values[0]), 'source': '판매 데이터'},
        {'element': '제품 가격 경쟁력', 'score': float(satisfaction_comparison[satisfaction_comparison['attribute'] == '가격']['satisfaction_gap'].values[0]), 'source': '고객 만족도 조사'},
        {'element': '제품 다양성', 'score': -0.35, 'source': '고객 리뷰 감성 분석'},
        {'element': '반품 및 환불 정책', 'score': -0.42, 'source': '고객 리뷰 감성 분석'}
    ],
    'opportunities': [
        {'element': '스마트홈 기기 시장 성장', 'score': float(market_data[market_data['category'] == '스마트홈']['market_growth'].values[0]), 'source': '시장 데이터'},
        {'element': '온라인 쇼핑 증가 트렌드', 'score': float(trend_analysis[trend_analysis['trend_name'] == '온라인 쇼핑 증가']['impact_score'].values[0]), 'source': '소비자 트렌드'},
        {'element': '친환경 제품 수요 증가', 'score': 0.65, 'source': '소비자 트렌드'},
        {'element': '5G 기술 확산', 'score': 0.58, 'source': '기술 트렌드'}
    ],
    'threats': [
        {'element': '대형 플랫폼의 시장 진입', 'score': -0.72, 'source': '경쟁 환경 분석'},
        {'element': '가격 경쟁 심화', 'score': float(market_data[market_data['category'] == '컴퓨터']['competition_intensity'].values[0]), 'source': '시장 데이터'},
        {'element': '공급망 불안정성', 'score': -0.68, 'source': '산업 리포트'},
        {'element': '경기 침체 가능성', 'score': float(trend_analysis[trend_analysis['trend_name'] == '경기 침체']['impact_score'].values[0]), 'source': '경제 전망'}
    ]
}

# SWOT 매트릭스 시각화
plt.figure(figsize=(14, 10))

# 서브플롯 구성 - wspace를 0.4로 증가시켜 좌우 간격 넓힘
gs = plt.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.4, hspace=0.3)

# 각 영역별 색상 정의
colors = {
    'strengths': 'green',
    'weaknesses': 'red',
    'opportunities': 'blue',
    'threats': 'orange'
}

# 강점(Strengths) 서브플롯
ax_s = plt.subplot(gs[0, 0])
elements = swot_elements['strengths']
elements.sort(key=lambda x: x['score'], reverse=True)  # 점수 기준 내림차순 정렬
y_pos = range(len(elements))
scores = [e['score'] for e in elements]
labels = [e['element'] for e in elements]
ax_s.barh(y_pos, scores, color=colors['strengths'], alpha=0.7)
ax_s.set_yticks(y_pos)
ax_s.set_yticklabels(labels)
ax_s.set_title('Strengths (강점)', fontweight='bold', color=colors['strengths'])
ax_s.set_xlim(0, max(scores) * 1.2)
for i, v in enumerate(scores):
    ax_s.text(v + 0.05, i, f"{v:.2f}", va='center')

# 약점(Weaknesses) 서브플롯
ax_w = plt.subplot(gs[0, 1])
elements = swot_elements['weaknesses']
elements.sort(key=lambda x: x['score'])  # 점수 기준 오름차순 정렬 (약점은 낮을수록 더 심각)
y_pos = range(len(elements))
scores = [abs(e['score']) for e in elements]  # 절대값으로 변환하여 표시
labels = [e['element'] for e in elements]
ax_w.barh(y_pos, scores, color=colors['weaknesses'], alpha=0.7)
ax_w.set_yticks(y_pos)
ax_w.set_yticklabels(labels)
ax_w.set_title('Weaknesses (약점)', fontweight='bold', color=colors['weaknesses'])
ax_w.set_xlim(0, max(scores) * 1.2)
for i, v in enumerate(scores):
    ax_w.text(v + 0.05, i, f"{v:.2f}", va='center')

# 기회(Opportunities) 서브플롯
ax_o = plt.subplot(gs[1, 0])
elements = swot_elements['opportunities']
elements.sort(key=lambda x: x['score'], reverse=True)  # 점수 기준 내림차순 정렬
y_pos = range(len(elements))
scores = [e['score'] for e in elements]
labels = [e['element'] for e in elements]
ax_o.barh(y_pos, scores, color=colors['opportunities'], alpha=0.7)
ax_o.set_yticks(y_pos)
ax_o.set_yticklabels(labels)
ax_o.set_title('Opportunities (기회)', fontweight='bold', color=colors['opportunities'])
ax_o.set_xlim(0, max(scores) * 1.2)
for i, v in enumerate(scores):
    ax_o.text(v + 0.05, i, f"{v:.2f}", va='center')

# 위협(Threats) 서브플롯
ax_t = plt.subplot(gs[1, 1])
elements = swot_elements['threats']
elements.sort(key=lambda x: x['score'])  # 점수 기준 오름차순 정렬 (위협은 낮을수록 더 심각)
y_pos = range(len(elements))
scores = [abs(e['score']) for e in elements]  # 절대값으로 변환하여 표시
labels = [e['element'] for e in elements]
ax_t.barh(y_pos, scores, color=colors['threats'], alpha=0.7)
ax_t.set_yticks(y_pos)
ax_t.set_yticklabels(labels)
ax_t.set_title('Threats (위협)', fontweight='bold', color=colors['threats'])
ax_t.set_xlim(0, max(scores) * 1.2)
for i, v in enumerate(scores):
    ax_t.text(v + 0.05, i, f"{v:.2f}", va='center')

plt.suptitle('데이터 기반 계량화된 SWOT 분석 - 디지털마트', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('quantified_swot_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- TOWS 전략 생성 및 LLM(OpenAI) 활용 ---
print("\n--- 7. TOWS 전략 생성 및 LLM(OpenAI) 활용 시작 ---")

# 1. LLM(OpenAI) API 호출 함수 (Exponential Backoff 적용)
def call_openai_llm(prompt, strategy_type, retries=3, initial_delay=5):
    """OpenAI API를 호출하여 특정 전략 타입에 대한 제안을 생성합니다. (Exponential Backoff 적용)"""
    if not openai_client:
        return f"LLM({strategy_type}) 기본 응답: API 키 없음. 프롬프트: {prompt[:50]}..."

    instruction = ""
    if strategy_type == 'SO': instruction = "이 강점과 기회를 활용하여 구체적이고 실행 가능한 '성장 전략' 아이디어를 1~2문장으로 제안해주세요..."
    elif strategy_type == 'ST': instruction = "이 강점을 활용하여 주어진 위협에 효과적으로 대응하거나 그 영향을 최소화할 수 있는 '차별화 또는 방어 전략' 아이디어를 1~2문장으로 제안해주세요..."
    elif strategy_type == 'WO': instruction = "이 기회를 활용하여 주어진 약점을 극복하거나 개선할 수 있는 '역량 강화 또는 전환 전략' 아이디어를 1~2문장으로 제안해주세요..."
    elif strategy_type == 'WT': instruction = "이 약점과 위협 상황을 고려하여 부정적 영향을 최소화하거나 회피할 수 있는 '축소, 회피, 또는 방어적 전략' 아이디어를 1~2문장으로 제안해주세요..."

    full_prompt = f"{prompt}\n\n{instruction}"
    current_delay = initial_delay

    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "당신은 비즈니스 전략 전문가입니다. 주어진 SWOT 요소를 바탕으로 간결하고 실행 가능한 전략을 한국어로 제안합니다."},
                    {"role": "user", "content": full_prompt},
                ],
                max_tokens=150,
                temperature=0.7,
            )

            if response and response.choices:
                result_text = response.choices[0].message.content.strip().replace('\n', ' ')
                print(f"    OpenAI 응답 수신 성공 (시도 {attempt+1})")
                return f"LLM({strategy_type}): {result_text}"
            else:
                print(f"    Warning (시도 {attempt+1}): OpenAI가 유효한 텍스트 응답을 반환하지 않았습니다.")
                if attempt == retries - 1:
                    return f"LLM({strategy_type}) 응답 오류: 빈 응답. 프롬프트: {prompt[:50]}..."

        except Exception as e:
            print(f"    Error (시도 {attempt+1}): OpenAI API 호출 중 오류 발생: {e}")
            if "429" in str(e) or "Rate limit" in str(e) or "rate_limit" in str(e):
                wait_time = current_delay * (2 ** attempt) + np.random.uniform(0, 1)
                print(f"      Rate limit 도달 가능성. {wait_time:.2f}초 후 재시도...")
                time.sleep(wait_time)
                if attempt == retries - 1:
                    print(f"      최대 재시도 ({retries}회) 후에도 Rate limit 오류 지속.")
                    return f"LLM({strategy_type}) 오류: Rate limit 지속 ({e}). 프롬프트: {prompt[:50]}..."
            else:
                if attempt == retries - 1:
                    return f"LLM({strategy_type}) 오류: API 호출 실패 ({e}). 프롬프트: {prompt[:50]}..."
                else:
                    print(f"      기타 오류 발생. {initial_delay / 2}초 후 재시도...")
                    time.sleep(initial_delay / 2)

    return f"LLM({strategy_type}) 오류: 최대 재시도 실패. 프롬프트: {prompt[:50]}..."


# 2. 점수 정규화 함수
def normalize_scores(swot_elements):
    print("  점수 정규화 중...")
    scaler = MinMaxScaler(feature_range=(0, 1))
    elements_copy = {cat: [item.copy() for item in items] for cat, items in swot_elements.items()}
    for category, items in elements_copy.items():
        valid_scores = [item['score'] for item in items if pd.notna(item.get('score'))]
        if not valid_scores:
            for item in items: item['norm_score'] = np.nan
            continue
        if len(valid_scores) == 1:
            norm_value = 0.5
            for item in items:
                 if pd.notna(item.get('score')): item['norm_score'] = norm_value
                 else: item['norm_score'] = np.nan
            continue
        try:
            scores_array = np.array(valid_scores).reshape(-1, 1)
            normalized_scores_map = dict(zip(valid_scores, scaler.fit_transform(scores_array).flatten()))
            for item in items:
                score = item.get('score')
                if pd.notna(score): item['norm_score'] = normalized_scores_map.get(score, np.nan)
                else: item['norm_score'] = np.nan
        except Exception as e:
            print(f"    '{category}' 정규화 중 오류: {e}. NaN으로 설정.")
            for item in items: item['norm_score'] = np.nan
    print("  점수 정규화 완료.")
    return elements_copy

# 3. 내부/외부 점수 계산 함수
def calculate_scores(normalized_swot_elements):
    print("  내부/외부 종합 점수 계산 중...")
    scores = {}
    for category, items in normalized_swot_elements.items():
        valid_norm_scores = [item['norm_score'] for item in items if pd.notna(item.get('norm_score'))]
        if not valid_norm_scores: scores[category] = 0.0
        else:
            if category in ['weaknesses', 'threats']: scores[category] = np.mean([abs(s) for s in valid_norm_scores])
            else: scores[category] = np.mean(valid_norm_scores)
    internal_score = scores.get('strengths', 0.0) - scores.get('weaknesses', 0.0)
    external_score = scores.get('opportunities', 0.0) - scores.get('threats', 0.0)
    print(f"  계산된 내부 점수: {internal_score:.2f}, 외부 점수: {external_score:.2f}")
    return internal_score, external_score

# 4. TOWS 전략 생성 함수
def generate_tows_strategies(normalized_swot_elements):
    """SWOT 요소를 조합하고 OpenAI LLM을 호출하여 TOWS 전략을 생성합니다."""
    print("  TOWS 전략 생성 및 OpenAI LLM 호출 중...")
    strategies = {'SO': [], 'ST': [], 'WO': [], 'WT': []}
    valid_elements = {cat: [item for item in items if pd.notna(item.get('norm_score'))]
                      for cat, items in normalized_swot_elements.items()}
    call_interval = 1.0
    print(f"    API 호출 간격: {call_interval} 초")

    total_calls = sum(len(valid_elements.get(c1, [])) * len(valid_elements.get(c2, []))
                      for c1, c2 in [('strengths', 'opportunities'), ('strengths', 'threats'),
                                     ('weaknesses', 'opportunities'), ('weaknesses', 'threats')])
    print(f"    예상 총 API 호출 횟수: {total_calls}")
    call_count = 0

    print("  SO 전략 생성 시작...")
    for s, o in product(valid_elements.get('strengths', []), valid_elements.get('opportunities', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) SO 조합 처리 중...")
        prompt = f"강점: {s['element']} (정규화 점수: {s['norm_score']:.2f}), 기회: {o['element']} (정규화 점수: {o['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'SO')
        strategies['SO'].append({'description': llm_response, 'impact_score': (s['norm_score'] + o['norm_score']) / 2, 'feasibility': np.random.uniform(0.6, 0.9), 'category': 'SO'})
        time.sleep(call_interval)

    print("  ST 전략 생성 시작...")
    for s, t in product(valid_elements.get('strengths', []), valid_elements.get('threats', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) ST 조합 처리 중...")
        prompt = f"강점: {s['element']} (정규화 점수: {s['norm_score']:.2f}), 위협: {t['element']} (정규화 점수: {t['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'ST')
        strategies['ST'].append({'description': llm_response, 'impact_score': (s['norm_score'] + abs(t['norm_score'])) / 2, 'feasibility': np.random.uniform(0.5, 0.8), 'category': 'ST'})
        time.sleep(call_interval)

    print("  WO 전략 생성 시작...")
    for w, o in product(valid_elements.get('weaknesses', []), valid_elements.get('opportunities', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) WO 조합 처리 중...")
        prompt = f"약점: {w['element']} (정규화 점수: {w['norm_score']:.2f}), 기회: {o['element']} (정규화 점수: {o['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'WO')
        strategies['WO'].append({'description': llm_response, 'impact_score': (abs(w['norm_score']) + o['norm_score']) / 2, 'feasibility': np.random.uniform(0.4, 0.7), 'category': 'WO'})
        time.sleep(call_interval)

    print("  WT 전략 생성 시작...")
    for w, t in product(valid_elements.get('weaknesses', []), valid_elements.get('threats', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) WT 조합 처리 중...")
        prompt = f"약점: {w['element']} (정규화 점수: {w['norm_score']:.2f}), 위협: {t['element']} (정규화 점수: {t['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'WT')
        strategies['WT'].append({'description': llm_response, 'impact_score': (abs(w['norm_score']) + abs(t['norm_score'])) / 2, 'feasibility': np.random.uniform(0.3, 0.6), 'category': 'WT'})
        time.sleep(call_interval)

    print("  TOWS 전략 생성 완료.")
    return strategies

# 5. 전략 우선순위 평가 함수
def prioritize_strategies(strategies):
    print("  전략 우선순위 평가 중...")
    strategy_list = [
        {**strategy, 'priority_score': strategy.get('impact_score', 0) * strategy.get('feasibility', 0)}
        for category in strategies for strategy in strategies[category]
    ]
    if not strategy_list:
        print("    생성된 전략이 없어 우선순위를 평가할 수 없습니다.")
        return pd.DataFrame()
    strategy_df = pd.DataFrame(strategy_list)
    strategy_df = strategy_df.sort_values('priority_score', ascending=False)
    print("  전략 우선순위 평가 완료.")
    return strategy_df

# --- 실행 ---
# 1. 점수 정규화
normalized_swot = normalize_scores(swot_elements)
# 2. 내부/외부 점수 계산
internal_score, external_score = calculate_scores(normalized_swot)
# 3. TOWS 전략 생성 (OpenAI 사용, 시간 소요 예상)
generated_strategies = generate_tows_strategies(normalized_swot)
# 4. 우선순위 평가
strategy_df = prioritize_strategies(generated_strategies)

print("--- 7. TOWS 전략 생성 및 LLM(OpenAI) 활용 완료 ---")

In [ ]:
# --- 결과 시각화 및 출력 ---
print("\n--- 8. 최종 결과 시각화 및 요약 시작 ---")

def visualize_strategies(strategy_df, internal_score, external_score):
    """전략적 위치를 시각화하고 우선순위가 높은 전략을 출력합니다."""
    # 전략적 위치 시각화
    plt.figure(figsize=(10, 8))
    limit_x = max(abs(internal_score), 0.1) * 1.5
    limit_y = max(abs(external_score), 0.1) * 1.5

    plt.scatter(internal_score, external_score, s=300, color='C4', zorder=5, label=f'현재 위치 ({internal_score:.2f}, {external_score:.2f})', marker='*')
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
    plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
    plt.text(internal_score, external_score + limit_y*0.05, f"({internal_score:.2f}, {external_score:.2f})", fontsize=10, ha='center', va='bottom')

    # 사분면 텍스트
    plt.text(limit_x*0.5, limit_y*0.5, "SO 전략\n(공격적)", fontsize=12, ha='center', va='center', color='green', alpha=0.7)
    plt.text(-limit_x*0.5, limit_y*0.5, "WO 전략\n(회복)", fontsize=12, ha='center', va='center', color='blue', alpha=0.7)
    plt.text(limit_x*0.5, -limit_y*0.5, "ST 전략\n(다각화)", fontsize=12, ha='center', va='center', color='orange', alpha=0.7)
    plt.text(-limit_x*0.5, -limit_y*0.5, "WT 전략\n(방어적)", fontsize=12, ha='center', va='center', color='red', alpha=0.7)

    plt.title('전략적 위치 분석 (TOWS)', fontsize=16, fontweight='bold')
    plt.xlabel(f'내부 역량 점수 (정규화 기반)', fontsize=12)
    plt.ylabel(f'외부 환경 점수 (정규화 기반)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xlim(-limit_x, limit_x)
    plt.ylim(-limit_y, limit_y)
    plt.legend(loc='upper right')
    plt.tight_layout()
    # plt.savefig('tows_strategic_position.png', dpi=300)
    plt.show()

    # 우선순위 전략 출력
    if not strategy_df.empty:
        print("\nLLM 기반 우선순위 전략 제안 (상위 5개):")
        # description이 너무 길 경우 줄여서 표시
        strategy_df['description_short'] = strategy_df['description'].str[:100] + '...'
        # 표시할 컬럼 선택 및 포맷팅
        display_df = strategy_df[['category', 'description_short', 'priority_score', 'impact_score', 'feasibility']].head(5)
        # 점수 소수점 2자리까지 표시
        pd.options.display.float_format = '{:.2f}'.format
        print(display_df.to_markdown(index=False))
        # 원상 복구 (선택적)
        pd.reset_option('display.float_format')


        print("\n*참고:")
        print("  - 'priority_score' = 'impact_score' * 'feasibility'. 상대적 우선순위를 나타냅니다.")
    else:
        print("\n생성된 전략이 없어 우선순위를 표시할 수 없습니다.")

# 시각화 및 결과 출력 실행
# 이전 셀에서 계산된 strategy_df, internal_score, external_score 사용
if 'strategy_df' in locals() and 'internal_score' in locals() and 'external_score' in locals():
     visualize_strategies(strategy_df, internal_score, external_score)
else:
     print("오류: 전략 데이터 또는 내부/외부 점수가 계산되지 않아 결과를 표시할 수 없습니다.")

print("\n--- 8. 최종 결과 시각화 및 요약 완료 ---")
print("\n=== 전체 프로세스 종료 ===")

# SWOT 분석 결과 해석

## 분석 과정 설명

제시된 코드는 SWOT 분석(강점-약점-기회-위협)을 바탕으로 TOWS 매트릭스를 작성하고 전략적 위치를 시각화하는 과정을 보여줍니다. 이 분석은 다음과 같은 단계로 진행되었습니다:

1. **내부 역량과 외부 환경 점수 계산**:
   - 내부 역량 점수(internal_score): 강점(S)과 약점(W) 요인을 종합하여 계산
   - 외부 환경 점수(external_score): 기회(O)와 위협(T) 요인을 종합하여 계산

2. **전략적 위치 시각화**:
   - 산출된 내부 역량 점수와 외부 환경 점수를 좌표 평면에 표시
   - 사분면에 따라 각기 다른 전략적 접근법을 구분(SO, WO, ST, WT)

3. **우선순위 전략 제안**:
   - 각 전략 유형별로 우선순위 점수(priority_score) 계산
   - 영향력(impact_score)과 실현 가능성(feasibility)을 곱하여 산출
   - 상위 5개 우선순위 전략을 표로 제시

## 결과 해석

### 우선순위 전략 분석

출력된 결과에서 상위 5개 우선순위 전략을 살펴보면:

1. **최우선 전략(ST)**:
   - 우선순위 점수: 0.735618
   - 내용: 고성장하는 모바일 전자기기 시장에서 프리미엄급 제품 라인업을 강화하거나, 차별화된 소프트웨어/서비스를 제공하여 가격 경쟁에서 벗어나 고부가가치 전략을 추구
   - 특징: 내부 강점(모바일 전자기기 시장 성장)을 활용하여 외부 위협(가격 경쟁 심화)에 대응

2. **2순위 전략(SO)**:
   - 우선순위 점수: 0.614699
   - 내용: 모바일 전자기기 판매 채널을 활용하여 스마트홈 기기 시장에 진출, 기존 고객 기반을 통해 스마트홈 기기 패키지 상품을 제공하고 묶음 할인 및 연동 서비스 제공
   - 특징: 강점(모바일 전자기기 판매 채널)과 기회(스마트홈 시장 성장)를 결합한 공격적 전략

3. **3순위 전략(WT)**:
   - 우선순위 점수: 0.597066
   - 내용: 가전제품 시장의 가격 경쟁 심화에 대응하여 프리미엄급 제품 라인에 집중하거나 틈새시장 공략을 통해 차별화된 가치를 제공하는 축소 전략 채택
   - 특징: 약점(가전제품 부문)과 위협(가격 경쟁 심화)에 대한 방어적 전략

4. **4순위 전략(WO)**:
   - 우선순위 점수: 0.471261
   - 내용: 스마트홈 기기 시장 성장이라는 기회를 활용하여 IoT 기반의 스마트 가전 제품 개발 및 판매를 확대함으로써 가전제품 매출 성장률 저하라는 약점을 극복
   - 특징: 약점(가전제품 매출 성장률 저조)을 외부 기회(스마트홈 시장)를 통해 보완하는 회복 전략

5. **5순위 전략(SO)**:
   - 우선순위 점수: 0.431995
   - 내용: 모바일 전자기기 매출 성장세를 바탕으로 온라인 쇼핑 증가 트렌드에 맞춰 모바일 전자기기 전용 온라인 판매 채널을 강화하고 맞춤형 모바일 광고 및 사용자 경험 최적화
   - 특징: 비교적 낮은 영향력 점수(0.505213)이지만 매우 높은 실현 가능성(0.855075)을 가진 전략

## 종합적 해석

수정된 분석 결과는 해당 기업이 다음과 같은 상황에 있음을 시사합니다:

1. **모바일 전자기기 부문 강세**: 모바일 전자기기 시장에서 고성장을 보이며 이를 활용한 전략이 우선시됨

2. **가전제품 부문 약세**: 가전제품 시장에서는 매출 성장률이 저조하고 가격 경쟁에 취약한 상황

3. **시장 환경 변화**: 스마트홈 시장 성장과 온라인 쇼핑 증가라는 기회가 있는 반면, 가격 경쟁 심화라는 위협이 존재

이 기업의 전략적 방향성은 다음과 같이 요약할 수 있습니다:

1. **차별화 전략 우선**: 최우선 전략으로 가격 경쟁을 피하고 프리미엄화와 차별화를 통한 고부가가치 창출이 제시됨

2. **교차 판매 활용**: 모바일 전자기기의 강점을 활용해 스마트홈 시장으로 확장하는 시너지 전략이 높은 우선순위를 가짐

3. **약점 부문 전략적 재편**: 약점이 있는 가전제품 부문은 프리미엄화나 틈새시장 공략, 또는 스마트 가전으로의 전환을 통해 경쟁력 회복이 필요

4. **디지털 채널 강화**: 온라인 판매 채널 강화와 디지털 마케팅 최적화가 실현 가능성이 높은 전략으로 제시됨

이 기업은 모바일 전자기기 시장에서의 경쟁력을 기반으로 프리미엄화와 시장 확장을 추구하는 한편, 약점이 있는 가전제품 부문은 차별화와 스마트화를 통해 경쟁력을 회복하는 이중적 접근이 필요합니다. 특히 모바일과 스마트홈의 연계를 통한 생태계 구축이 중요한 전략적 방향으로 보입니다.